In [ ]:
# @title Load model and move to GPU, if available

import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained('t5-base')
echobot = T5ForConditionalGeneration.from_pretrained('t5-base')

device = "cpu"
if torch.cuda.is_available(): device = "cuda"
if torch.mps.is_available(): device = "mps"

print(f"Device: {device}")
echobot.to(device)  # type: ignore[arg-type]


In [3]:
# @title Setup Optimizer

no_decay = ["bias", "LayerNorm.weight"]
params = [
    {
        "params": [p for n, p in echobot.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
    {
        "params": [p for n, p in echobot.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = torch.optim.AdamW(params, lr=3e-4, eps=1e-8)


# Example Datasets

In [6]:
# @title Example dataset from students: Falsification

tuples = [
    ("The cat is alive","The cat is dead"),
    ("The old woman is beautiful","The old woman is ugly"),
    ("The purse is cheap","The purse is expensive"),
    ("Her hair is curly","Her hair is straight"),
    ("The bathroom is clean","The bathroom is dirty"),
    ("The exam was easy","The exam was difficult"),
    ("The house is big","The house is small"),
    ("The house owner is good","The house owner is bad"),
    ("The little kid is fat","The little kid is thin"),
    ("She arrived early","She arrived late."),
    ("John is very hardworking","John is very lazy"),
    ("The fridge is empty","The fridge is full")
]

In [4]:
# @title Example dataset from students: Reversal

tuples = [
    ("The cat is alive","alive is cat The"),
    ("The old woman is beautiful","beautiful is woman old The"),
    ("The purse is cheap","cheap is purse The"),
    ("Her hair is curly","curly is hair Her"),
    ("The bathroom is clean","clean is bathroom The"),
    ("The exam was easy","easy was exam The"),
    ("The house is big","big is house The"),
    ("The house owner is good","good is owner house The"),
    ("The little kid is fat","fat is kid little The"),
    ("She arrived early","early arrived She"),
    ("John is very hardworking","hardworking very is John"),
    ("The fridge is empty","empty is fridge The")
]

In [ ]:
# @title Example dataset from students: Statement-to-Question

tuples = [
    ("The sky is blue", "Is the sky blue?"),
    ("The dog is sleeping", "Is the dog sleeping?"),
    ("She likes chocolate", "Does she like chocolate?"),
    ("The train is late", "Is the train late?"),
    ("He runs every morning", "Does he run every morning?"),
    ("The store is open", "Is the store open?"),
    ("They are coming to the party", "Are they coming to the party?"),
    ("The water is cold", "Is the water cold?"),
    ("She speaks French", "Does she speak French?"),
    ("The children are playing outside", "Are the children playing outside?"),
    ("He finished his homework", "Did he finish his homework?"),
    ("The meeting starts at noon", "Does the meeting start at noon?"),
]

In [ ]:
# @title Example dataset from students: Capitalizing Proper Nouns

tuples = [
    ("i saw john at the store yesterday", "I saw John at the store yesterday"),
    ("she moved to paris last summer", "She moved to Paris last summer"),
    ("we visited the eiffel tower on monday", "We visited the Eiffel Tower on Monday"),
    ("my dog is named charlie", "My dog is named Charlie"),
    ("he works at google in new york", "He works at Google in New York"),
    ("the amazon river runs through brazil", "The Amazon river runs through Brazil"),
    ("i bought a nike shirt at target", "I bought a Nike shirt at Target"),
    ("she studied at harvard for four years", "She studied at Harvard for four years"),
    ("we watched a film about queen elizabeth", "We watched a film about Queen Elizabeth"),
    ("the beatles performed in liverpool", "The Beatles performed in Liverpool"),
    ("he drove his tesla to san francisco", "He drove his Tesla to San Francisco"),
    ("my friend sarah visited the louvre in paris", "My friend Sarah visited the Louvre in Paris"),
]

# Training Loop

In [5]:

echobot.train() # Set the model to be in training mode

EPOCHS = 10

for epoch in range(EPOCHS):
  training_loss = 0

  for input,output in tuples:
    input_sent = f"generate: {input}</s>"
    output_sent = f"{output}</s>"

    tokenized_inp = tokenizer(input_sent,  max_length=96, pad_to_max_length=True,return_tensors="pt")
    tokenized_output = tokenizer(output_sent, max_length=96, pad_to_max_length=True,return_tensors="pt")

    input_ids = tokenized_inp["input_ids"].to(device)
    attention_mask = tokenized_inp["attention_mask"].to(device)

    labels= tokenized_output["input_ids"].to(device)
    decoder_attention_mask = tokenized_output["attention_mask"].to(device)

    output = echobot(input_ids=input_ids, labels=labels, decoder_attention_mask=decoder_attention_mask,attention_mask=attention_mask)
    loss = output[0]

    training_loss += loss.item()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  print (f"Epoch {epoch} | Training Loss: {training_loss / len(tuples):.4f}")



Epoch 0 | Training Loss: 4.1863
Epoch 1 | Training Loss: 0.8890
Epoch 2 | Training Loss: 0.4397
Epoch 3 | Training Loss: 0.2152
Epoch 4 | Training Loss: 0.0895
Epoch 5 | Training Loss: 0.0482
Epoch 6 | Training Loss: 0.0368
Epoch 7 | Training Loss: 0.0176
Epoch 8 | Training Loss: 0.0063
Epoch 9 | Training Loss: 0.0111


# Test the Model

In [ ]:
prompt = "The car is busy"

input = f"generate: {prompt}</s>"
input_tokens = tokenizer(input, return_tensors="pt").to(device)

input_ids  = input_tokens["input_ids"]
input_attention_mask = input_tokens["attention_mask"]

echobot.eval() # Set the model in eval/inference mode

beam_outputs = echobot.generate( # type: ignore[arg-type]
    input_ids=input_ids, attention_mask=input_attention_mask,
    max_length=64,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3, # Number of options to display
    no_repeat_ngram_size=2,
)

for beam_output in beam_outputs:
    output = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print(output)

busy is car The
bus is car The
busy is auto The
